# Notebook 3: Hybrid Silo RAG & Vectorless Hierarchical Handbook RAG

## Two Retrieval Paradigms, One Project

This notebook implements a **dual-retrieval pipeline** that deliberately uses two different retrieval strategies on two different kinds of content — not to show off techniques, but because each strategy is the genuinely correct tool for its content type.

### Why Embedding-Based Search Breaks Down on Long Documents

Vector similarity works excellently for short, atomic text snippets where each chunk carries independent meaning. A 2-paragraph refund policy snippet is perfect for this.

But consider a **45-page Compliance Handbook** structured as:
- Chapter 1: Return Windows by Product Category  
- Chapter 2: Regional Tax & Duty Waivers  
- Chapter 3: GDPR Data Deletion + Refund Interactions  
- Chapter 4: Cross-Border Dispute Escalation Procedures  

When you chunk this and embed the pieces, three problems emerge:

| Problem | Why it matters |
|---|---|
| **Context fragmentation** | A paragraph in Ch3 references definitions in Ch1 — chunked independently, both lose meaning |
| **Wrong-chapter retrieval** | Query: "EU customer wants regional tax waiver on returned goods" — vector similarity may pull Ch1 ("returned goods" is textually close) instead of Ch2 (the right answer) |
| **Cross-reference blindness** | "See Section 4.2 for escalation" — the embedding of Section 4.2 is not linked to the section that references it |

### The Vectorless Tree Navigation Solution

Instead of chunking the handbook, we:
1. **Build once:** Parse the document into a hierarchical tree — title + LLM-generated summary per chapter
2. **Navigate per query:** Show the LLM the *summaries only* (the table of contents), let it reason about which chapter is relevant, then fetch only the full text of the selected chapter

This is how a human expert would use the same document — flip to the right chapter, not scan every paragraph for keyword overlap.

```
Query: "EU customer wants a regional tax waiver"
         │
         ▼
   [handbook_tree.json] (summaries only, ~1KB)
         │
         ▼  LLM reasons: "Ch2 covers regional tax & duty waivers"
         ▼
   [ch2_full_text.txt] loaded from disk (~8KB)
         │
         ▼
   Precise clause extracted
```

### Retrieval Decision Matrix

| | Vector RAG (Qdrant) | Vectorless RAG (Tree Navigation) |
|---|---|---|
| **Content** | Short policy snippets (refund windows, return rules) | 45-page Compliance Handbook (chapters, cross-refs) |
| **Query type** | "What is the 30-day return window?" | "Customer is invoking GDPR rights alongside a refund" |
| **Mechanism** | Dense + BM25 sparse hybrid search | LLM navigates chapter summaries → reads selected chapter |
| **LLM calls** | 0 per retrieval | 1-2 per query (navigate + extract) |
| **Latency** | ~50ms | ~2-4s |
| **When to use** | High volume, simple factual lookups | Rare, high-stakes, structurally complex queries |

In [ ]:
# Cell 1: Setup — create realistic synthetic policy documents and configure LLM client.
#
# We generate in-memory policy documents so this notebook runs without
# pre-existing files. In production these would be real PDFs or .txt files.

import os
import json
import pathlib
import textwrap
from dotenv import load_dotenv

project_root = pathlib.Path(".").resolve().parent
load_dotenv(project_root / ".env")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
GATEWAY_BASE_URL = "http://localhost:8000/v1"
GATEWAY_API_KEY = os.environ.get("LITELLM_MASTER_KEY", "sk-vaultrag-master-key")

import httpx

def check_gateway_live(url: str = GATEWAY_BASE_URL, timeout: float = 3.0) -> bool:
    try:
        resp = httpx.get(url.replace("/v1", "/health"), timeout=timeout)
        return resp.status_code == 200
    except Exception:
        return False

gateway_live = check_gateway_live()
if gateway_live:
    LLM_BASE_URL = GATEWAY_BASE_URL
    LLM_API_KEY = GATEWAY_API_KEY
    LLM_MODEL = "enterprise-llama"
    print("✅ Using LiteLLM Gateway")
else:
    LLM_BASE_URL = "https://api.groq.com/openai/v1"
    LLM_API_KEY = GROQ_API_KEY
    LLM_MODEL = "llama3-8b-8192"
    print("ℹ️  Gateway not live — using Groq API directly.")

if not LLM_API_KEY:
    raise EnvironmentError("GROQ_API_KEY not set. Add it to .env or export as env var.")

from openai import OpenAI

llm_client = OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)


def call_llm(prompt: str, max_tokens: int = 800) -> str:
    """Call the LLM gateway (or direct API) and return raw text response."""
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0,
    )
    return response.choices[0].message.content


def call_llm_json(prompt: str, max_tokens: int = 1024) -> dict:
    """Call LLM and parse response as JSON. Returns dict with error key on failure."""
    raw = call_llm(prompt, max_tokens=max_tokens)
    cleaned = raw.strip()
    # Strip markdown code fences if present
    for fence in ("```json", "```"):
        if cleaned.startswith(fence):
            cleaned = cleaned[len(fence):]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"error": "json_parse_failure", "raw": raw[:500]}


# Synthetic policy documents — two tiers (public and internal)
POLICY_DOCS = {
    "refund_policy_public": [
        {
            "filename": "standard_return_window.txt",
            "text": textwrap.dedent("""
                STANDARD RETURN & REFUND POLICY — PUBLIC

                Section 1: Return Window
                Customers may return most items within 30 calendar days of delivery.
                Electronics and appliances have a 15-day return window.
                Perishable goods are final sale and cannot be returned.

                Section 2: Refund Processing
                Approved refunds are processed within 5-7 business days.
                Original payment method is refunded by default.
                Store credit may be offered at customer request.

                Section 3: Condition Requirements
                Items must be returned in original packaging, unused.
                Items showing signs of use may receive a partial refund (up to 50%).
                Proof of purchase is required for all returns.
            """).strip(),
        },
        {
            "filename": "shipping_damage_policy.txt",
            "text": textwrap.dedent("""
                SHIPPING DAMAGE POLICY — PUBLIC

                Section 1: Reporting Timeframe
                Shipping damage must be reported within 48 hours of delivery.
                Photographic evidence is required.

                Section 2: Resolution Options
                Option A: Full replacement shipped at no charge within 3 business days.
                Option B: Full refund processed within 5-7 business days.
                Option C: Partial refund up to 75% if item remains functional.

                Section 3: Carrier Liability
                Claims against shipping carriers are handled by our logistics team.
                Customer does not need to contact the carrier directly.
            """).strip(),
        },
    ],
    "refund_policy_internal": [
        {
            "filename": "manager_discretion_exceptions.txt",
            "text": textwrap.dedent("""
                MANAGER DISCRETION EXCEPTION RULES — INTERNAL (SUPERVISOR/ADMIN ONLY)

                Section 1: Goodwill Refunds
                Supervisors may approve goodwill refunds up to $150 without VP sign-off.
                Refunds over $150 require written VP approval via the escalation form.

                Section 2: Outside-Window Exceptions
                Agents may grant 60-day return windows for loyalty customers (tier Gold+).
                Approval must be logged in the CRM with reason code EX-01.

                Section 3: Repeat Abuse Detection
                Customers with 3+ refund requests in 90 days are flagged for review.
                Do NOT communicate the flagging to the customer.
            """).strip(),
        },
    ],
}

print("✅ Synthetic policy documents created.")
for tier, docs in POLICY_DOCS.items():
    print(f"  • {tier}: {len(docs)} document(s)")

In [ ]:
# Cell 2: Vector RAG Pipeline — Qdrant + SentenceTransformer embeddings.
#
# Steps:
#   1. Connect to Qdrant (local Docker container on port 6333)
#   2. Create two Silo collections: policy_public, policy_internal
#   3. Chunk documents with RecursiveCharacterTextSplitter
#   4. Embed each chunk with SentenceTransformer('all-MiniLM-L6-v2') — local, no API
#   5. Upsert into Qdrant
#   6. Run a test query and display retrieved results

import uuid
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Qdrant client setup ──────────────────────────────────────────────────────
QDRANT_HOST = os.environ.get("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.environ.get("QDRANT_PORT", 6333))

try:
    qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT, timeout=10)
    qdrant_client.get_collections()  # Connectivity check
    print(f"✅ Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}")
    qdrant_live = True
except Exception:
    print("ℹ️  Qdrant not reachable — using in-memory mode for demo.")
    qdrant_client = QdrantClient(":memory:")
    qdrant_live = False

# ── Embedding model ──────────────────────────────────────────────────────────
print("Loading SentenceTransformer (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
VECTOR_DIM = 384  # all-MiniLM-L6-v2 output dimension
print("✅ Embedder ready.")

# ── Text splitter ─────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def ingest_collection(
    collection_name: str,
    documents: list[dict],
) -> list[dict]:
    """Create Qdrant collection, chunk docs, embed, and upsert. Returns doc metadata."""
    # (Re)create collection
    qdrant_client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
    )

    points: list[PointStruct] = []
    all_chunks: list[dict] = []
    point_id = 0

    for doc in documents:
        chunks = splitter.split_text(doc["text"])
        vectors = embedder.encode(chunks, batch_size=32, show_progress_bar=False)

        for chunk_text, vector in zip(chunks, vectors):
            chunk_meta = {
                "id": point_id,
                "text": chunk_text,
                "source": doc["filename"],
                "collection": collection_name,
            }
            all_chunks.append(chunk_meta)
            points.append(
                PointStruct(
                    id=point_id,
                    vector=vector.tolist(),
                    payload={"text": chunk_text, "source": doc["filename"]},
                )
            )
            point_id += 1

    if points:
        qdrant_client.upsert(collection_name=collection_name, points=points)

    return all_chunks


# Ingest both tiers
doc_texts_by_collection: dict[str, list[dict]] = {}

for tier, docs in POLICY_DOCS.items():
    collection = tier.replace("refund_policy_", "policy_")  # e.g. policy_public
    chunks = ingest_collection(collection, docs)
    doc_texts_by_collection[collection] = chunks
    print(f"✅ Ingested {len(chunks)} chunks into collection '{collection}'")


# ── Test retrieval ────────────────────────────────────────────────────────────
def dense_search(query: str, collection: str, top_k: int = 3) -> list[dict]:
    """Embed query and perform cosine similarity search in Qdrant."""
    vector = embedder.encode(query).tolist()
    hits = qdrant_client.search(
        collection_name=collection,
        query_vector=vector,
        limit=top_k,
        with_payload=True,
    )
    return [{"score": h.score, "text": h.payload["text"], "source": h.payload["source"]} for h in hits]


TEST_QUERY = "What is the return window for electronics?"
results = dense_search(TEST_QUERY, "policy_public", top_k=3)

print(f"\n🔍 Test Query: {TEST_QUERY!r}")
print(f"   Top {len(results)} results from 'policy_public':")
for i, r in enumerate(results, 1):
    print(f"   [{i}] score={r['score']:.3f} | source={r['source']}")
    print(f"       {r['text'][:150].strip()}...")

In [ ]:
# Cell 3: Vectorless RAG — Build the Compliance Handbook Tree.
#
# This cell simulates the "build once at ingestion" phase:
#   1. Generate a realistic 40-page mock compliance handbook as structured text
#   2. Parse chapter headers
#   3. For each chapter: call LiteLLM/Groq to generate a JSON summary node
#   4. Assemble a hierarchical tree and save to handbook_tree.json
#
# In production: step 1 is replaced by reading a real PDF via pdfplumber/PyMuPDF.

import re

# ── Mock 40-page Compliance Handbook ─────────────────────────────────────────
COMPLIANCE_HANDBOOK_RAW = """
COMPLIANCE HANDBOOK — VERSION 4.2
VaultRAG Customer Operations Division

===== CHAPTER 1: STANDARD RETURN WINDOWS BY PRODUCT CATEGORY =====
All return windows begin at the confirmed delivery date, not the order date.
General Merchandise: 30 calendar days. Electronics and appliances: 15 calendar days.
Software licenses and digital downloads: non-refundable after activation.
Perishable and customized items: final sale, no returns accepted.
Exceptions to standard windows require supervisor approval and must be logged in CRM
with exception code EX-01. High-volume customers (Gold tier and above) may receive
extended 60-day windows per manager discretion. All return window exceptions must be
documented and are subject to quarterly audit.

===== CHAPTER 2: REGIONAL TAX AND DUTY WAIVERS =====
EU customers subject to VAT (Value Added Tax) may request a full VAT waiver on
returned goods when the return is confirmed within the statutory 14-day cooling-off
period under EU Consumer Rights Directive 2011/83/EU. US customers in tax-exempt
states (e.g., Oregon, Montana) receive automatic tax waiver on all eligible returns.
Cross-border returns originating from outside the original purchase country incur
import duty reclaim handled by our customs team — customer receives net refund.
UK post-Brexit returns: VAT reclaim follows HMRC Schedule 9 procedures.
All tax waivers require a valid tax invoice number submitted via the Returns Portal.

===== CHAPTER 3: GDPR DATA DELETION AND REFUND INTERACTIONS =====
Under GDPR Article 17 (Right to Erasure), a customer's refund request does NOT
automatically trigger data deletion. The two processes are legally separate.
A customer may request erasure after the refund is complete. The company retains
transaction records for 7 years for tax compliance purposes (GDPR Recital 65 exception).
During an active refund dispute, account data cannot be erased — processing must
complete first. Supervisors must escalate concurrent erasure + refund requests to
the Data Protection Officer (DPO) within 48 hours. GDPR erasure requests must be
fulfilled within 30 days of completion of the refund dispute.

===== CHAPTER 4: CROSS-BORDER DISPUTE ESCALATION PROCEDURES =====
Disputes involving cross-border transactions (purchase and delivery in different
countries) must follow a three-stage escalation:
Stage 1 (Days 1-5): Support agent gathers evidence — order confirmation, delivery
proof, customer complaint details. Stage 2 (Days 6-10): Supervisor reviews and
determines jurisdiction. EU disputes default to consumer's country of residence.
Stage 3 (Days 11-30): Legal review if resolution not reached. External mediator
engaged for disputes over USD $500. All cross-border disputes are logged in the
Global Dispute Register (GDR) within 24 hours of first customer contact.
UK/EU disputes post-Brexit may require dual-jurisdiction analysis.

===== CHAPTER 5: SHIPPING CARRIER LIABILITY AND DAMAGE CLAIMS =====
Shipping damage must be reported within 48 hours of confirmed delivery.
Carriers are liable for damage occurring in transit. Evidence required:
photographic proof of damage, original packaging intact. For claims under $100:
support agent may approve direct replacement without carrier claim submission.
For claims $100-$500: supervisor approval required, carrier claim filed within 72 hours.
For claims over $500: legal review, third-party assessment required.
Carrier claim resolution SLA is 15 business days. Customer is refunded from company
funds first; carrier reimbursement is internal accounting.

===== CHAPTER 6: LEGAL DISCLAIMERS AND REGULATORY COMPLIANCE =====
This handbook is for internal use only and does not constitute legal advice.
All policies are subject to applicable local consumer protection laws which
supersede internal policy where they provide greater consumer rights.
Regulatory changes affecting refund obligations are reviewed quarterly by Legal.
Country-specific overrides: Australia (ACL), Canada (provincial consumer protection),
Japan (Consumer Contract Act) may grant customers rights exceeding standard policy.
Support agents are not authorized to make binding legal representations.
All legal matters must be escalated to the Legal department via ticket type LEGAL-01.
""".strip()


def parse_chapters(handbook_text: str) -> dict[str, str]:
    """Split handbook into chapters by detecting === CHAPTER N: TITLE === headers."""
    pattern = re.compile(r"={3,}\s*(CHAPTER\s+\d+[^=]+)\s*={3,}", re.IGNORECASE)
    sections: dict[str, str] = {}

    matches = list(pattern.finditer(handbook_text))
    for i, match in enumerate(matches):
        chapter_id = f"Ch{i + 1}"
        chapter_title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(handbook_text)
        chapter_text = handbook_text[start:end].strip()
        sections[chapter_id] = {"title": chapter_title, "text": chapter_text}

    return sections


def build_chapter_summary_node(chapter_id: str, chapter_data: dict) -> dict:
    """Ask LLM to summarize a chapter into a structured JSON node."""
    prompt = f"""You are building a hierarchical table-of-contents tree for a Compliance Handbook.

Analyze this chapter and produce a JSON summary node. Be concise — the summary
will be shown to an LLM later to decide which chapter to read.

Chapter ID: {chapter_id}
Chapter Title: {chapter_data['title']}
Chapter Text:
{chapter_data['text']}

Return ONLY valid JSON with this exact structure:
{{
  "id": "{chapter_id}",
  "title": "<chapter title>",
  "summary": "<2-3 sentence summary of what this chapter covers and what questions it answers>",
  "keywords": ["<keyword1>", "<keyword2>", "<keyword3>"]
}}"""

    result = call_llm_json(prompt, max_tokens=400)
    if "error" in result:
        # Fallback if LLM fails to produce valid JSON
        return {
            "id": chapter_id,
            "title": chapter_data["title"],
            "summary": chapter_data["text"][:200],
            "keywords": [],
        }
    return result


# Parse and build tree
chapters = parse_chapters(COMPLIANCE_HANDBOOK_RAW)
print(f"📖 Parsed {len(chapters)} chapters from the Compliance Handbook.")

handbook_tree: dict = {
    "document": "VaultRAG Compliance Handbook v4.2",
    "version": "4.2",
    "total_chapters": len(chapters),
    "children": [],
}

# Store full chapter texts separately for later retrieval (never passed to LLM as-is)
CHAPTER_FULL_TEXTS: dict[str, str] = {}

for chapter_id, chapter_data in chapters.items():
    print(f"  Summarizing {chapter_id}: {chapter_data['title'][:50]}...")
    CHAPTER_FULL_TEXTS[chapter_id] = chapter_data["text"]
    node = build_chapter_summary_node(chapter_id, chapter_data)
    handbook_tree["children"].append(node)

# Save to handbook_tree.json in project root
tree_path = project_root / "handbook_tree.json"
with open(tree_path, "w", encoding="utf-8") as f:
    json.dump(handbook_tree, f, indent=2)

print(f"\n✅ handbook_tree.json saved to: {tree_path}")
print("\n--- Tree Structure Preview ---")
for child in handbook_tree["children"]:
    print(f"  [{child['id']}] {child['title']}")
    print(f"        Summary: {child.get('summary', '')[:100]}...")

In [ ]:
# Cell 4: Vectorless RAG — Navigation and Extraction Class.
#
# HandbookNavigator:
#   navigate()  — Show LLM only the high-level tree (titles + summaries).
#                 LLM reasons which chapter(s) are relevant, returns node IDs.
#   retrieve()  — Fetch only the full text of selected chapters from disk/memory.
#   extract()   — Ask LLM to pull the precise clause from the selected text.
#
# No vector search. No embeddings. Just LLM reasoning over a structured map.

from dataclasses import dataclass
from typing import Optional


@dataclass
class HandbookResult:
    """Result object from a HandbookNavigator.answer() call."""
    query: str
    selected_chapter_ids: list[str]
    navigation_reasoning: str
    retrieved_texts: dict[str, str]  # chapter_id → full text
    extracted_answer: str
    role_filtered: bool


# Role-based chapter visibility (same principle as Qdrant Silo collections)
ROLE_HANDBOOK_CHAPTERS = {
    "support_agent": {"Ch1", "Ch2", "Ch3", "Ch5"},    # No legal (Ch6) or escalation (Ch4)
    "supervisor":    {"Ch1", "Ch2", "Ch3", "Ch4", "Ch5"},   # + cross-border escalation
    "admin":         {"Ch1", "Ch2", "Ch3", "Ch4", "Ch5", "Ch6"},  # Everything
}


class HandbookNavigator:
    """Vectorless RAG navigator for the Compliance Handbook.

    The LLM reasons over chapter SUMMARIES (not full text) to decide which
    chapter is relevant. Full text is only loaded for selected chapters.
    Role-based filtering happens on the TREE shown to the LLM — restricted
    chapters are structurally absent, not filtered from output.
    """

    def __init__(
        self,
        tree: dict,
        full_texts: dict[str, str],
        llm_caller=call_llm_json,
        text_caller=call_llm,
    ) -> None:
        self.tree = tree
        self.full_texts = full_texts
        self._call_json = llm_caller
        self._call_text = text_caller

    def _filter_tree_by_role(self, role: str) -> dict:
        """Return a copy of the tree with restricted chapters removed entirely.
        The LLM never sees a restricted chapter's title or summary —
        it structurally cannot navigate somewhere it was never shown.
        """
        allowed = ROLE_HANDBOOK_CHAPTERS.get(role, set())
        visible_children = [
            c for c in self.tree.get("children", []) if c["id"] in allowed
        ]
        return {**self.tree, "children": visible_children}

    def navigate(self, query: str, visible_tree: dict) -> dict:
        """LLM navigation step: reason over summaries, select relevant chapter IDs."""
        # Build compact representation (titles + summaries only, NOT full text)
        toc_compact = json.dumps(
            [
                {
                    "id": c["id"],
                    "title": c["title"],
                    "summary": c.get("summary", ""),
                    "keywords": c.get("keywords", []),
                }
                for c in visible_tree.get("children", [])
            ],
            indent=2,
        )

        prompt = f"""You are navigating a compliance handbook to answer a support query.

Below is the table of contents (chapter IDs, titles, and summaries only).
Select the chapter IDs most relevant to answering the query.

Table of Contents:
{toc_compact}

Query: "{query}"

Think step by step: which chapter(s) contain the answer? Select at most 2.

Return ONLY valid JSON:
{{"node_ids": ["Ch1", "Ch2"], "reasoning": "<explain which chapter and why>"}}"""

        return self._call_json(prompt, max_tokens=512)

    def retrieve_chapters(self, chapter_ids: list[str]) -> dict[str, str]:
        """Load full text of selected chapters from storage.
        Only selected chapter text is passed to extraction — keeping token
        usage proportional to what's actually relevant.
        """
        return {
            cid: self.full_texts[cid]
            for cid in chapter_ids
            if cid in self.full_texts
        }

    def extract_answer(self, query: str, chapter_texts: dict[str, str]) -> str:
        """Given the full text of selected chapters, extract the precise answer."""
        combined = "\n\n".join(
            f"[{cid}]\n{text}" for cid, text in chapter_texts.items()
        )
        prompt = f"""Extract the precise clause or rule that answers the following query.
Quote directly from the text where possible. If the answer is not in the text, say so.

Handbook text:
{combined}

Query: {query}

Answer:"""
        return self._call_text(prompt, max_tokens=500)

    def answer(self, query: str, role: str = "admin") -> HandbookResult:
        """Full vectorless RAG pipeline: filter → navigate → retrieve → extract."""
        # Step 1: Filter tree to what this role is allowed to see
        visible_tree = self._filter_tree_by_role(role)
        role_filtered = len(visible_tree["children"]) < len(
            self.tree.get("children", [])
        )

        # Step 2: LLM navigates the visible table of contents
        nav_result = self.navigate(query, visible_tree)
        selected_ids: list[str] = nav_result.get("node_ids", [])
        reasoning: str = nav_result.get("reasoning", "No reasoning provided.")

        # Step 3: Load full text of selected chapters only
        retrieved = self.retrieve_chapters(selected_ids)

        # Step 4: Extract precise answer from retrieved text
        answer = (
            self.extract_answer(query, retrieved)
            if retrieved
            else "No relevant chapter found in the allowed sections for this role."
        )

        return HandbookResult(
            query=query,
            selected_chapter_ids=selected_ids,
            navigation_reasoning=reasoning,
            retrieved_texts=retrieved,
            extracted_answer=answer,
            role_filtered=role_filtered,
        )


# ── Instantiate and test the navigator ──────────────────────────────────────
navigator = HandbookNavigator(tree=handbook_tree, full_texts=CHAPTER_FULL_TEXTS)

TEST_CASES = [
    {
        "query": "Customer wants a regional tax waiver on their EU return",
        "role": "support_agent",
        "description": "Tax waiver query — should navigate to Ch2",
    },
    {
        "query": "Customer is invoking GDPR data deletion rights alongside their refund complaint",
        "role": "admin",
        "description": "GDPR + refund query — should navigate to Ch3",
    },
    {
        "query": "Cross-border dispute from EU customer buying from US store, what is the escalation procedure?",
        "role": "support_agent",
        "description": "Escalation query as support_agent — Ch4 is restricted, should find nothing",
    },
]

print("=" * 70)
print("  VECTORLESS RAG NAVIGATION TESTS")
print("=" * 70)

for i, tc in enumerate(TEST_CASES, 1):
    print(f"\n[Test {i}] {tc['description']}")
    print(f"  Role:  {tc['role']}")
    print(f"  Query: {tc['query'][:80]}...")

    result = navigator.answer(query=tc["query"], role=tc["role"])

    print(f"  Selected chapters: {result.selected_chapter_ids}")
    print(f"  Navigation reasoning: {result.navigation_reasoning[:200]}")
    print(f"  Role filtered: {result.role_filtered}")
    print(f"  Extracted answer (first 300 chars):")
    print(f"    {result.extracted_answer[:300]}")

print("\n✅ Vectorless RAG tests complete.")